# ONNX Relax 测试

In [ ]:
path = "/media/pc/data/board/arria10/lxw/tasks/tools/npuusertools/models/test/nanotrack_head/nanotrack_head.onnx"

In [10]:
import onnx
import tvm
from tvm import relax
from tvm.relax.frontend.onnx import from_onnx
model = onnx.load(path)

mod = from_onnx(model)

In [11]:
mod.show()

In [12]:
fold_pipeline = tvm.transform.Sequential([
    relax.transform.FoldBatchnormToConv2D(),
    relax.transform.FoldConstant(),
    relax.transform.RemoveRedundantReshape(),
    # 将 BatchNorm 转换为一组更简单的算子以进行融合
    relax.transform.DecomposeOpsForInference(),
    # 规范化绑定
    relax.transform.CanonicalizeBindings(),

])
# 初次优化模型
run_mod = fold_pipeline(mod)

In [13]:
run_mod.show()